# Object Tracking — Colab Runner (3 cells)

**Local:** `bash colab/pack_bundle.sh` → produces `colab/bundle.zip` (code + kaggle.json).
**Colab:** Runtime → GPU, then run cells top-to-bottom.

- Cell 1 — upload `bundle.zip` (single prompt) → unpacks code, installs Kaggle creds + deps
- Cell 2 — downloads data from Kaggle (~650 MB, once per session)
- Cell 3 — runs the tracking pipeline and downloads the output

In [ ]:
# Cell 1 — upload bundle.zip (tracking/ + kaggle.json)
import os, shutil, zipfile
from google.colab import files

WORK = '/content/work'
if os.path.isdir(os.path.join(WORK, 'tracking')):
    shutil.rmtree(os.path.join(WORK, 'tracking'))
os.makedirs(WORK, exist_ok=True)

up = files.upload()
zname = next(iter(up))
with zipfile.ZipFile(zname) as zf:
    zf.extractall(WORK)
os.remove(zname)

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move(os.path.join(WORK, 'kaggle.json'), '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('code tree:', sorted(os.listdir(os.path.join(WORK, 'tracking'))))
!pip install kaggle
!pip install -r /content/work/tracking/requirements.txt
!nvidia-smi | head -20


In [ ]:
# Cell 2 — download data from Kaggle (once per session)
import os
DATA = '/content/data'
if not os.path.exists(os.path.join(DATA, 'images')):
    os.makedirs(DATA, exist_ok=True)
    !kaggle datasets download -d sharma37/object-tracking-data -p {DATA} --unzip
print('data tree:', sorted(os.listdir(DATA)))

In [ ]:
# Cell 3 — run tracking, download result, disconnect runtime
import os
VIDEO = '/content/data/images/NVR-ABD6_ch50_main_20260409092100_20260409110059.mp4'
OUT_DIR = '/content/work/results'
os.makedirs(OUT_DIR, exist_ok=True)
OUTPUT = os.path.join(OUT_DIR, 'NVR-ABD6_ch50_main_20260409092100_20260409110059_output_r50.mp4')
assert os.path.exists(VIDEO), VIDEO

%cd /content/work
!PYTHONPATH=/content/work python -m tracking.run_tracking "$VIDEO" \
  --output "$OUTPUT" \
  --frames 0 --stride 2 --threshold 0.7 --ocr-every 1 \
  --min-vehicle-area 8000 --min-ocr-conf 0.7 --lock-in-conf 0.95

from google.colab import files, runtime
size_mb = os.path.getsize(OUTPUT) / 1024 / 1024
print(f'[cell3] downloading {size_mb:.1f} MB → browser…')
files.download(OUTPUT)

input('\n[cell3] Press Enter AFTER the file finishes downloading in your browser to disconnect the runtime…')
print('[cell3] disconnecting runtime…')
runtime.unassign()
